In [252]:
from typing import Iterable, Iterator, Callable
from itertools import chain as iterchain, combinations as itercomb
from functools import reduce
from collections import Counter
from pprint import pprint

iterflat = iterchain.from_iterable

In [253]:
from board import DIGITS, Digits, Loc, Cell, Board

In [254]:
def boardiff(b0: Board, b2: Board) -> Iterable[Cell]:
    """Compare boards, yield removed content"""
    for c0, c2 in zip(iter(b0), iter(b2)):
        if c0.digits != c2.digits:
            yield Cell(c0.loc, Digits(c0.digits - c2.digits))


# Topology

The board is classicaly 9 rows, 9 columns, and 9 boxes over them. These are major units.

Intersection of a row and a column gives individual cell. A cell is subdivided further into 9 segments for digits, but only for visualizing purposes.

Intersection of a box with a row or column gives sector.

Generalized, all the localities can be addressed by a set of `(box, col, row)`

Visibility of 2 localities means they share some unit (or two). It determines location of cell peers and possibly conflicting drafts.


In [255]:
from topology import Zone, peers, peerz, allpeers, visibility, allvisible

In [256]:
# construction
assert Zone.B(1) == Zone(1, ..., ...)
assert Zone.R(2) == Zone(..., 2, ...)
assert Zone.C(3) == Zone(..., ..., 3)
assert Zone.L(Loc(2, 3)) == Zone(1, 2, 3)  # fulfiled box

assert Zone(1, 2, 3).is_cell
assert Zone(1, ..., ...).is_unit
assert Zone(1, 2, ...).is_sector

# intersection
assert Zone.R(2) & Zone.C(3) == Zone(1, 2, 3)
assert Zone.B(1) & Zone.R(2) == Zone(1, 2, ...)
assert Zone.B(1) & Zone.C(3) == Zone(1, ..., 3)

# containering
assert set(iter(Zone(1, ..., ...))) == {
    Loc(1, 1),
    Loc(1, 2),
    Loc(1, 3),
    Loc(2, 1),
    Loc(2, 2),
    Loc(2, 3),
    Loc(3, 1),
    Loc(3, 2),
    Loc(3, 3),
}
assert set(iter(Zone(1, 2, ...))) == {Loc(2, 1), Loc(2, 2), Loc(2, 3)}
assert Loc(2, 3) in Zone.B(1)
assert Loc(2, 3) in Zone.R(2)
assert Loc(2, 3) in Zone.C(3)

In [257]:
assert visibility(Zone.L(Loc(1, 2)), Zone.L(Loc(2, 3))) == {Zone.B(1)}
assert visibility(Zone.L(Loc(1, 2)), Zone.L(Loc(1, 3))) == {Zone.R(1), Zone.B(1)}
assert visibility(Zone(1, 2, ...), Zone.L(Loc(2, 9))) == {Zone.R(2)}
assert visibility(Zone.L(Loc(1, 2)), Zone.L(Loc(8, 9))) == set()

assert allvisible(Zone.L(Loc(1, 2)), Zone.L(Loc(8, 9))) == {Zone(3, 1, 9), Zone(7, 8, 2)}
assert allvisible(Zone.L(Loc(1, 2)), Zone.L(Loc(2, 3))) == {Zone(1, ..., ...)}
assert allvisible(Zone.L(Loc(1, 2)), Zone.L(Loc(3, 2))) == {Zone(1, ..., ...), Zone(..., ..., 2)}

In [258]:
print(*map(str, Zone.Units()))

b1…… b2…… b3…… b4…… b5…… b6…… b7…… b8…… b9…… …r1… …r2… …r3… …r4… …r5… …r6… …r7… …r8… …r9… ……c1 ……c2 ……c3 ……c4 ……c5 ……c6 ……c7 ……c8 ……c9


In [259]:
assert peers(Loc(1, 2), Zone.B(1)) == {
    Loc(1, 1),
    Loc(1, 3),
    Loc(2, 1),
    Loc(2, 2),
    Loc(2, 3),
    Loc(3, 1),
    Loc(3, 2),
    Loc(3, 3),
}
assert peerz(Zone(1, 2, ...), Zone.B(1)) == {
    Loc(1, 1),
    Loc(1, 2),
    Loc(1, 3),
    Loc(3, 1),
    Loc(3, 2),
    Loc(3, 3),
}

# Targeting

Generic draft target:

- location: cell, unit, sector
- digits: one or more digits


In [260]:
from analysis import Node, cellmatching, cellspoiling

In [261]:
assert Node.C(Cell(Loc(1, 2), Digits({1, 2, 3}))) == Node(Zone.L(Loc(1, 2)), Digits({1, 2, 3}))
assert Node.at(Loc(1, 2), Digits({1, 2, 3})) == Node(Zone.L(Loc(1, 2)), Digits({1, 2, 3}))
assert Node.at(Loc(1, 2), 5) == Node(Zone.L(Loc(1, 2)), Digits({5}))

assert Node(Zone.L(Loc(1, 2)), Digits({1})).is_cellular
assert Node(Zone.L(Loc(1, 2)), Digits({1})).loc == Loc(1, 2)
assert Node(Zone.L(Loc(1, 2)), Digits({1})).is_singular
assert Node(Zone.L(Loc(1, 2)), Digits({1})).is_casual
assert not Node(Zone.L(Loc(1, 2)), Digits({1, 2})).is_singular
assert not Node(Zone.L(Loc(1, 2)), Digits({1, 2})).is_casual
assert not Node(Zone(1, 2, ...), Digits({1})).is_cellular
assert not Node(Zone(1, 2, ...), Digits({1})).is_casual

# Resolving

searching for patterns -> generating resolutions -> applying resolutions


In [262]:
from utils import filt_finals, filt_having, filt_havesome, draftborhood, draftboard, flat_cells, count_finals, count_digits, grab_digits
from solving import Resolver, Resolving, Pattern, solver, onceolver
from searching import search_breadth
from analysis import Link, HLink, SLink, Chain

In [263]:
async def solve_silent(initial: Board, *resolvers: Resolver):
    result = initial
    _, _, drafted = result.validate()
    assert drafted
    async for _, _, result in solver(initial, *resolvers):
        complete, valid, drafted = result.validate()
        if complete or not valid or not drafted:
            break
    return result

In [264]:
async def solve_logging(initial: Board, /, *resolvers: Resolver, filtout: set[str] | None = None):
    result = initial
    iterations = 0
    _, _, drafted = result.validate()
    assert drafted
    current = result
    async for resolver, pattern, result in solver(initial, *resolvers):
        iterations += 1
        if filtout and resolver.__name__ in filtout:
            continue
        print(f"{iterations:03d} {resolver.__name__}", end=": ")
        diff = list(boardiff(current, result))
        print("-", " ".join(map(str, diff)))
        # pprint(rule)

        current = result
        complete, valid, drafted = result.validate()
        if complete or not valid or not drafted:
            break

    complete, valid, drafted = result.validate()
    stuck = not complete and not drafted
    print("========")
    print(f"{iterations=} {complete=} {valid=} {stuck=}")
    return result


# Resolvers/Rules


## Random choice


In [265]:
def random_choice(board: Board) -> Resolving:
    drafts = list(draftboard(board))
    assert len(drafts)

    drafts.sort(key=lambda c: len(c))
    leastcell = drafts[0]
    counts = count_digits(drafts)

    digits = list(leastcell.digits)
    digits.sort(key=lambda d: counts[d])
    leastdig = digits[0]

    chosen = Node.at(leastcell, leastdig)
    loosen = Node.at(leastcell, leastcell.digits - chosen.digits)

    yield Pattern(
        anchors={chosen},
        spoilers={loosen},
    )

## Rectangles

TODO


## Singles


In [266]:
def naked_singles(board: Board) -> Resolving:
    for fincell in filter(filt_finals, iter(board)):
        findig = Digits({fincell.final})
        anchor = Node.at(fincell, findig)
        spoilers = set(filter(filt_havesome(findig), board.slice(allpeers(anchor.zone))))
        if spoilers:
            yield Pattern(
                anchors={anchor},
                spoilers={Node.at(c.loc, anchor.digits) for c in spoilers},
            )

In [267]:
def hidden_singles(board: Board) -> Resolving:
    for unit in Zone.Units():
        drafts = set(draftborhood(board, unit))
        counts = count_digits(drafts)
        for dig, cnt in counts.items():
            if cnt == 1:
                [cell] = filter(lambda c: dig in c, drafts)
                anchor = Node.at(cell, dig)
                spoilers = Node.at(cell, cell.digits - {dig})
                yield Pattern(
                    space={Node.at(unit, dig)},
                    anchors={anchor},
                    spoilers={spoilers},
                )


## Multiples


In [268]:
def naked_multiples(board: Board) -> Resolving:
    for unit in Zone.Units():
        drafts = tuple(draftborhood(board, unit))
        digits = grab_digits(drafts)
        for m in range(2, 5):
            for combo in itercomb(digits, m):
                combits = Digits(combo)
                habitat = set(filter(lambda c: c.digits & combits, drafts))
                naked = set(filter(lambda c: c.digits <= combits, habitat))
                # print(unit, combits, set(map(str, naked)), "/", set(map(str, habitat)))
                if len(naked) == len(combo):
                    spoilers = set(filter(filt_havesome(combits), drafts)) - naked
                    if spoilers:
                        yield Pattern(
                            space={Node.at(unit, combits)},
                            anchors={Node.at(c, combits) for c in naked},
                            spoilers={Node.at(c, combits) for c in spoilers},
                        )

In [269]:
def hidden_multiples(board: Board) -> Resolving:
    for unit in Zone.Units():
        drafts = tuple(draftborhood(board, unit))
        digits = grab_digits(drafts)
        for m in range(2, 5):
            for combo in itercomb(digits, m):
                combits = Digits(combo)
                habitat = set(filter(lambda c: c.digits & combits, drafts))
                if len(habitat) == len(combo):
                    spoilers = set(filter(lambda c: c.digits > combits, habitat))
                    if spoilers:
                        yield Pattern(
                            space={Node.at(unit, combits)},
                            anchors={Node.at(c, c.digits & combits) for c in habitat},
                            spoilers={Node.at(c, c.digits - combits) for c in spoilers},
                        )

## Triplets


In [21]:
def iter_sect() -> Iterator[tuple[Zone, Zone, Zone]]:
    for box in Zone.Allbox():
        for side in Zone.across(box):
            yield box, side, box & side  # type: ignore impossible null

In [22]:
def locked_triplets(board: Board) -> Resolving:
    for box, side, sect in iter_sect():
        boxcounts = count_digits(draftborhood(board, box))
        sidecounts = count_digits(draftborhood(board, side))
        sectcounts = count_digits(draftborhood(board, sect))
        # print(box, side, sect, sectcounts)
        for dig, cnt in sectcounts.items():
            if cnt < 2:
                continue
            elif cnt == boxcounts[dig] and cnt < sidecounts[dig]:
                sideborhood = set(iter(side)) - set(iter(box))
                yield Pattern(
                    anchors={Node.at(sect, dig)},
                    space={Node.at(box, dig)},
                    spoilers={Node.at(z, dig) for z in sideborhood},
                )
            elif cnt == sidecounts[dig] and cnt < boxcounts[dig]:
                insideborhood = set(iter(box)) - set(iter(side))
                yield Pattern(
                    anchors={Node.at(sect, dig)},
                    space={Node.at(side, dig)},
                    spoilers={Node.at(z, dig) for z in insideborhood},
                )


## Links/Chains


### Strong/Hard links

Criteria:

- (bi-location) only 2 drafts of same digit in a unit
- (bi-value) only 2 drafts in a cell


In [23]:
def search_hard_1(board: Board) -> Iterable[HLink]:
    """search for biloc single-value links"""
    for zone in Zone.Units():
        drafts = tuple(draftborhood(board, zone))
        counts = Counter(flat_cells(drafts))
        for dig, cnt in counts.items():
            if cnt == 2:
                (n1, n2) = filter(filt_having(dig), drafts)
                yield HLink((Node.at(n1, dig), Node.at(n2, dig)))

In [24]:
def search_hard_2(board: Board):
    """search for bivalue links"""
    for cell in filter(lambda c: len(c) == 2, draftboard(board)):
        d1, d2 = cell.digits
        yield HLink((Node.at(cell, d1), Node.at(cell, d2)))

- any 2 (non-overlapping) triplets that partition all habitants in a unit
- singular triplets may be involved in ajacent linking, when shrinked to normal singlular node


In [25]:
def search_hard_31(board: Board):
    """search for triplets and singles"""
    for unit in Zone.Units():
        drafts = set(draftborhood(board, unit))

        def subcount(dig: int):
            for sect in Zone.sectors(unit):
                subhab = tuple(filter(lambda c: dig in c and c.loc in sect, drafts))
                habcnt = len(subhab)
                if habcnt > 1:
                    yield Node.at(sect, dig), habcnt
                elif habcnt == 1:
                    yield Node.at(subhab[0], dig), habcnt

        habitants = count_digits(drafts)
        # print(unit, habitants)
        for dig, cnt in habitants.items():
            subcounts = {n: cnt for n, cnt in subcount(dig)}  # collapsing overlapping singular nodes
            for h1, h2 in itercomb(subcounts.keys(), 2):
                if not (h1.zone & h2.zone) and subcounts[h1] + subcounts[h2] == cnt:  # full partition
                    yield HLink((h1, h2))

### Weak/Soft links

Criteria:

- (bi-val) any 2 draft in a cell
- (bi-loc) any 2 drafts of same digit in a unit
- (bi-tripl) any 2 triplets in a unit == hard link
- (triploc) triplet and any singular draft of the same digit within shared unit


In [26]:
def softability_2(n1: Node, n2: Node):
    """check if the nodes are bival-soft-linkable"""
    return n1.dig != n2.dig and n1.is_cellular and n2.is_cellular and n1.loc == n2.loc


def softlink_2(n1: Node, n2: Node) -> SLink | None:
    """bi-value soft link if possible"""
    assert n1 != n2

    if softability_2(n1, n2):
        return SLink((n1, n2))

In [27]:
def softability_13(n1: Node, n2: Node):
    """check if the nodes are biloc-soft-linkable (both cellular and sectoral)"""
    return n1.dig == n2.dig and not (n1.zone & n2.zone) and len(visibility(n1.zone, n2.zone)) > 0


def softlink_13(n1: Node, n2: Node) -> SLink | None:
    """inter-location, both singulars and triplets"""
    assert n1 != n2

    if softability_13(n1, n2):
        return SLink((n1, n2))

In [28]:
def softlink_123(n1: Node, n2: Node) -> SLink | None:
    return softlink_2(n1, n2) or softlink_13(n1, n2)

In [29]:
Connecting = Callable[[Node, Node], SLink | None]

## Chains

Alternating inference chains, with simple bilocation links

X-Wing, X-Cycle, Nice Loop, etc


### Searching

- searching for all hard inks first
- trying to connect them into chains vith soft links


In [30]:
def expand_chain(current: Chain, links: Iterable[HLink], connecting: Connecting) -> Iterable[Chain]:
    """Expand chain to one of other hard links in the pool"""

    def close(chain):
        e1, e2 = chain.edges
        if len(chain) > 1 and chain.is_hardend:
            closing = connecting(e2, e1)
            if closing:
                yield Chain.extend(chain, closing)

    def extend(chain, link):
        e1, e2 = chain.edges
        x1, x2 = link
        if conn := connecting(e2, x1):
            yield Chain.extend(chain, conn, link)
        if conn := connecting(e2, x2):
            yield Chain.extend(chain, conn, link.reversed())
        if conn := connecting(x2, e1):
            yield Chain.extendhead(chain, link, conn)
        if conn := connecting(x1, e1):
            yield Chain.extendhead(chain, link.reversed(), conn)

    anchors: set[Node] = current.anchors()

    def noncycling(lnk: Link):
        return all(n.zone & a.zone is None for a in anchors for n in lnk)

    for link in filter(noncycling, links):
        for extended in extend(current, link):
            yield from close(extended)  # yield closed before open for breadth-first
            yield extended

In [31]:
def search_chains(current: Board, links: Iterable[HLink], matching: Callable[[Chain], bool], connecting: Connecting, max_length: int = 8):
    counts = count_finals(current)

    def rate(link: Link):
        return min(counts[link[0].dig], counts[link[1].dig])

    links = sorted(links, key=rate)  # prioritize most present (least final-counted)
    init = [Chain((l,)) for l in links]

    def expanding(chain: Chain):
        yield from expand_chain(chain, links, connecting)

    def canceling(chain: Chain):
        return len(chain) >= max_length

    yield from search_breadth(init, expanding, matching, canceling)

### Resolving


In [32]:
def softvision_2(board: Board, eye: Node) -> set[Node]:
    """All other digits in the cell are obviously visible"""
    if not eye.is_casual:
        return set()
    viscell = board.get(eye.loc)
    return {Node.at(viscell, d) for d in viscell.digits - eye.digits}

In [33]:
def softvision_13(board: Board, eye: Node) -> set[Node]:
    """All peer drafts visible and linkable from the eye node (either cell or sector)"""
    if not eye.is_singular:
        return set()
    dig = eye.dig
    visborhood = set(draftborhood(board, allpeers(eye.zone)))
    visbornood = {Node.at(c, dig) for c in visborhood if dig in c}
    return set(filter(lambda n: softability_13(n, eye), visbornood))

In [34]:
def crossvision_123(board: Board, eye1: Node, eye2: Node) -> set[Node]:
    e1vision = softvision_13(board, eye1) | softvision_2(board, eye1)
    e2vision = softvision_13(board, eye2) | softvision_2(board, eye2)
    return e1vision & e2vision

Pattern: a unclosed alternating hard-ended chain with matching edges

Rule: invalidate all visible from both edges


In [35]:
def match_rope(chain: Chain):
    e1, e2 = chain.edges
    if len(chain) > 2 and len(chain) % 2 == 1 and chain.is_hardend:
        if e1.dig != e2.dig:
            return e1.is_cellular and e2.is_cellular and e1.loc == e2.loc
        else:
            return e1.zone & e2.zone is None
    else:
        return False


def resolve_rope(board: Board, chain: Chain) -> Pattern | None:
    """Cleanup all spoilers visible from both edges"""
    e1, e2 = chain.edges
    spoilers = crossvision_123(board, e1, e2) - chain.anchors()
    if len(spoilers):
        return Pattern(
            spoilers=spoilers,
            anchors={e1, e2},
            chain=chain,
        )

#### loop

Pattern: a closed alternating chain (odd number of links)

Rule: invalidate all visible from both edges of each soft link


In [36]:
def match_loop(chain: Chain):
    return len(chain) > 2 and len(chain) % 2 == 0 and chain.is_loop


def resolve_loop(board: Board, chain: Chain) -> Pattern | None:
    """Cleanup all spoilers visible from both edges of each soft link"""
    links = tuple(filter(lambda lnk: isinstance(lnk, SLink), chain))
    spoilers = reduce(lambda a, b: a | b, (crossvision_123(board, l[0], l[1]) for l in links))
    spoilers -= chain.anchors()
    if len(spoilers):
        return Pattern(
            spoilers=spoilers,
            chain=chain,
        )

In [37]:
def match_usefull(chain: Chain):
    return match_loop(chain) or match_rope(chain)

#### resolver


In [251]:
def chains_(types: str):
    def conecting(n1: Node, n2: Node) -> SLink | None:
        if "2" in types:
            return softlink_2(n1, n2) or softlink_13(n1, n2)
        else:
            return softlink_13(n1, n2)

    def resolver(current: Board) -> Resolving:
        links = set()
        if "2" in types:
            links |= set(search_hard_2(current))
        if "3" in types:
            links |= set(search_hard_31(current))
        elif "1" in types:
            links |= set(search_hard_1(current))

        for chain in search_chains(current, links, matching=match_usefull, connecting=conecting, max_length=8):
            if match_loop(chain):
                res = resolve_loop(current, chain)
            elif match_rope(chain):
                res = resolve_rope(current, chain)
            if res:
                yield res
                break  # the search is infinite

    resolver.__name__ = f"chains[{types}]"

    return resolver

# A puzzle


In [273]:
from utils import fillempty, picture, parsepic, parsepic_wide

puzzle = parsepic("""
....89...
......17.
6........
.2.3.....
.1......9
.......68
8.9.5....
...7..2..
5........
""")

puzzle = Board.transform(puzzle, fillempty)

In [271]:
puzzle = await solve_silent(puzzle, naked_singles, hidden_singles)

In [274]:
puzzle = await solve_logging(
    puzzle,
    naked_singles,
    onceolver(hidden_singles),
    naked_multiples,
    onceolver(hidden_multiples),
    onceolver(locked_triplets),
    chains_("123"),
    # randomchoice,
    filtout={"naked_singles", "hidden_singles"},
)

022 hidden_multiples: - 5689@r1c1 12689@r1c2 2689@r1c3 3789@r1c4 12789@r1c7 126789@r1c8 1789@r1c9 15678@r2c1 1267@r2c2 12679@r2c3 13789@r2c4 15789@r2c5 1789@r2c6 1789@r2c9 126@r3c2 269@r3c3 36789@r3c4 5689@r3c5 689@r3c6 1234567@r3c7 1234567@r3c8 16789@r3c9 123568@r4c1 1239@r4c3 2358@r4c5 239@r4c6 123689@r4c7 236789@r4c8 23689@r4c9 125689@r5c1 129@r5c3 12379@r5c4 123589@r5c5 1239@r5c6 12689@r5c7 13456789@r5c8 12568@r6c1 1268@r6c2 12689@r6c3 3678@r6c4 3568@r6c5 3689@r6c6 12689@r6c7 12589@r7c2 35789@r7c4 5789@r7c6 12589@r7c7 256789@r7c8 2589@r7c9 256789@r8c1 125789@r8c2 25789@r8c3 2578@r8c5 2579@r8c6 267@r8c8 2789@r8c9 12589@r9c2 13456789@r9c3 2357@r9c4 2578@r9c5 2579@r9c6 125@r9c7 2567@r9c8 2589@r9c9
023 naked_multiples: - 89@r3c2 8@r3c3
062 naked_multiples: - 1347@r1c1 5@r1c2 345@r1c3 25@r1c4 25@r1c9 234@r2c1 3459@r2c2 58@r2c3 246@r2c4 56@r2c6 5@r2c9 347@r3c2 345@r3c3 5@r3c4 5@r3c6 8@r3c7 9@r3c8 5@r3c9 9@r4c1 4567@r4c3 1467@r4c5 14578@r4c6 5@r4c9 34578@r5c3 456@r5c4 6@r5c5 4678@r5c6 5@r

In [ ]:
links = set(search_hard_31(puzzle)) | set(search_hard_2(puzzle))
searching = search_chains(puzzle, links, match_usefull, connecting=softlink_123)

for _ in searching:
    pprint(_)

# GUI


In [42]:
%%html
<!-- fuck vscode -->
<style>
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-input-color: var(--vscode-editor-foreground);
    --jp-widgets-input-background-color: var(--vscode-editor-background);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
.jupyter-widgets input {
   background-color: var(--jp-widgets-input-background-color);
}
.cell-output-ipywidget-background {
   background-color: transparent !important;
}
</style>

In [43]:
import asyncio
from ipywidgets import widgets as w
from ipycanvas import hold_canvas
from IPython.display import display

from traitlets import HasTraits, Instance, Set, Unicode, observe, Bool, Dict
from canvas import SudokuCanvas

In [44]:
def click_future(button: w.Button) -> asyncio.Future[bool]:
    button.disabled = False
    future = asyncio.Future()

    def handle(b):
        button.on_click(handle, remove=True)
        button.disabled = True
        future.set_result(True)

    button.on_click(handle)

    return future


# TODO: make it cancellable somehow

In [45]:
class GUI(HasTraits):
    """Meta-widget with reactive properties and awaitable buttons"""

    puzzle = Instance(Board)
    status = Unicode()
    counters: Instance[Counter[int]] = Instance(Counter)

    # highlighting stuff
    targets = Set(Instance(Node))
    empties = Set(Instance(Node))
    anchors = Set(Instance(Node))
    links = Set(Instance(Link))

    # async running stuff
    running = Bool(False)
    paused = Bool(False)
    resolving = Unicode()
    inspecting = Dict(Bool(), Unicode(), default_value={})

    def __init__(self):
        super().__init__()
        self._canvas = SudokuCanvas()

        self._counters = {
            str(dig): w.Label(
                str(dig),
                layout=dict(width="auto", justify_content="center"),
                style=dict(text_color="black", background="var(--jp-info-color0)"),
            )
            for dig in DIGITS
        }
        self._counters["TOTAL"] = w.Label(
            "...",
            layout=dict(width="auto", justify_content="center"),
            style=dict(text_color="black", background="var(--jp-info-color0)"),
        )
        self._status = w.Label(
            layout=dict(width="auto", justify_content="center"),
            style=dict(text_color="black", background="var(--jp-info-color0)"),
        )
        #
        self._running = w.Button(
            icon="play",
            style=dict(
                font_size="large",
                text_color="var(--jp-info-color0)",
                button_color="transparent",
            ),
            tooltip="not a button",
        )
        self._continue = w.Button(description="Continue", disabled=True, button_style="primary")
        self._continue.layout.visibility = "hidden"
        #
        self._inspecting = w.VBox([w.Label("Inspecting"), w.VBox()])
        self._inspecting.layout.visibility = "hidden"
        self._resolving = w.Label()

    def _repr_mimebundle_(self, **kwargs):
        return w.HBox(
            [
                w.VBox(
                    [w.Label("Status"), *self._counters.values(), self._status],
                    layout=dict(align_items="stretch", width="7em"),
                ),
                self._canvas,
                w.VBox([
                    self._running,
                    self._resolving,
                    self._continue,
                    self._inspecting,
                ]),
            ],
            layout=dict(justify_content="flex-start", align_items="stretch"),
        )._repr_mimebundle_(**kwargs)

    @observe("puzzle")
    def upd_puzzle(self, change):
        self._canvas.draw_board(self.puzzle)
        self._hlayers = set()
        self.counters = count_finals(self.puzzle)

    def toggle_layer(self, layer: int):
        if not self.puzzle:
            return
        if layer in self._hlayers:
            self._hlayers.remove(layer)
        else:
            self._hlayers.add(layer)
        self._canvas.draw_board(self.puzzle, self._hlayers)

    @observe("status")
    def upd_status(self, change):
        value = self.status
        self._status.value = value
        self._status.style.visibility = "visible" if value != "" else "hidden"
        if value == "SOLVED":
            self._status.style.background = "var(--jp-success-color0)"
        elif value == "BROKEN":
            self._status.style.background = "var(--jp-error-color0)"
        else:
            self._status.style.background = "var(--jp-info-color0)"

    @observe("counters")
    def upd_counter(self, change):
        counters = self.counters
        for dig, cnt in counters.items():
            w = self._counters[str(dig)]
            w.value = f"{dig}: ({cnt})"
        total = counters.total()
        w = self._counters["TOTAL"]
        w.value = f"Total: ({total})"

    @observe("targets", "anchors", "empties", "links")
    def redraw_highlights(self, change):
        # redrawing everything in proper order
        self._canvas.clear_highlights()
        with hold_canvas():
            for node in self.empties:
                self._highlight_node(node, "pink")

            for lnk in self.links:
                self._highlight_link(lnk, "blue")
            for lnk in self.links:
                for n in lnk:
                    if n.is_cellular:
                        self._highlight_node(n, "blue")
                    else:
                        self._highlight_group(n, "blue")

            for node in self.anchors:
                if node.is_cellular:
                    self._highlight_node(node, "cyan")
                else:
                    self._highlight_group(node, "cyan")

            for cell in self.targets:
                self._highlight_node(cell, "red")

    def reset_highlights(self):
        self._canvas.clear_highlights()
        self.targets = set()
        self.anchors = set()
        self.empties = set()
        self.links = set()

    def _highlight_node(self, node: Node, color: str):
        for loc in node.zone:
            for dig in node.digits:
                self._canvas.highlight_segment(loc, dig, color=color)

    def _highlight_cell(self, cell: Cell, color: str):
        for dig in cell.digits:
            self._canvas.highlight_segment(cell.loc, dig, color=color)

    def _highlight_link(self, lnk: Link, color: str):
        n1, n2 = lnk
        if n1.zone.is_cell and n2.zone.is_cell:
            self._canvas.highlight_link(
                n1.loc,
                n1.dig,
                n2.loc,
                n2.dig,
                style=LINK_STYLES[lnk.__class__.__name__],
                color=color,
            )
        else:
            n1locs = tuple(iter(n1.zone))
            l1mid = Loc(
                sum(l.r for l in n1locs) // len(n1locs),
                sum(l.c for l in n1locs) // len(n1locs),
            )
            n2locs = tuple(iter(n2.zone))
            l2mid = Loc(
                sum(l.r for l in n2locs) // len(n2locs),
                sum(l.c for l in n2locs) // len(n2locs),
            )
            self._canvas.highlight_link(
                l1mid,
                n1.dig,
                l2mid,
                n2.dig,
                style=LINK_STYLES[lnk.__class__.__name__],
                color=color,
            )

    def _highlight_group(self, node: Node, color: str):
        locs = tuple(iter(node.zone))
        lmin = Loc(min(l.r for l in locs), min(l.c for l in locs))
        lmax = Loc(max(l.r for l in locs), max(l.c for l in locs))
        self._canvas.highlight_link(lmin, node.dig, lmax, node.dig, style="GROUP", color=color)
        self._highlight_node(node, color)

    @observe("running", "paused")
    def upd_running(self, change):
        self._running.icon = "play" if not self.running else "gear" if self.paused else "gear spin"
        self._running.disabled = self.running
        self._continue.layout.visibility = "visible" if self.running else "hidden"
        self._continue.disabled = not self.paused

    @observe("inspecting")
    def upd_inspecting(self, change):
        checkboxes = self._inspecting.children[1]
        for ch in checkboxes.children:
            ch.close()
        checkboxes.children = []
        if len(self.inspecting):
            checkboxes.children = [w.Checkbox(value=v, description=k, indent=False) for k, v in self.inspecting.items()]
            for ch in checkboxes.children:
                ch.observe(self.upd_inspecting_item, "value")
            self._inspecting.layout.visibility = "visible"
        else:
            self._inspecting.layout.visibility = "hidden"

    def upd_inspecting_item(self, change):
        checkbox = change["owner"]
        self.inspecting[checkbox.description] = checkbox.value

    @observe("resolving")
    def upd_resolving(self, change):
        self._resolving.value = self.resolving

    async def pause(self):
        self.paused = True
        await click_future(self._continue)
        self.paused = False


# GUI meta-widget


LINK_STYLES = {
    "Link": "SOLID",
    "HLink": "HARD",
    "SLink": "SOFT",
}

In [46]:
debug_view = w.Output()
gui = GUI()

In [ ]:
async def solve_ui(initial: Board, *resolvers: Resolver, filtout: set[str] = set()):
    current = initial
    result = initial

    _, _, drafted = result.validate()
    assert drafted

    gui.puzzle = current
    gui.running = True
    gui.inspecting = {r.__name__: r.__name__ not in filtout for r in resolvers}

    try:
        iteration = 0
        async for resolver, pattern, result in solver(initial, *resolvers):
            iteration += 1
            # print(iteration, resolver.__name__)
            # pprint(pattern)
            if gui.inspecting[resolver.__name__]:
                gui.puzzle = current
                resolving = f"#{iteration} {resolver.__name__}: ..."
                gui.resolving = resolving
                render_resolution(pattern)
                await gui.pause()
                clear_resolution()
                gui.puzzle = result
                await asyncio.sleep(0.2)
            current = result
            gui.resolving = f"#{iteration}"

            complete, valid, drafted = result.validate()
            render_status(complete, valid, drafted)
            if complete or not valid or not drafted:
                break
    except Exception as e:
        # FIXME: the cancel button handler
        with debug_view:
            raise RuntimeError("Solver failed") from e

    complete, valid, drafted = result.validate()
    render_status(complete, valid, drafted)
    gui.resolving = f"#{iteration} ENDED"
    gui.puzzle = result
    gui.running = False
    gui.inspecting = {}

    return result


def render_resolution(res: Pattern):
    with hold_canvas():
        gui.targets = res.get("spoilers", set())
        gui.anchors = res.get("anchors", set())
        gui.empties = res.get("space", set())
        if "chain" in res:
            gui.links = set(res["chain"])
        elif "links" in res:
            gui.links = res["links"]
        else:
            gui.links = set()


def render_status(complete: bool, valid: bool, drafted: bool):
    if not valid:
        gui.status = "BROKEN"
    elif complete:
        gui.status = "SOLVED"
    elif not drafted:
        gui.status = "STUCK"
    else:
        gui.status = "..."


def clear_resolution():
    gui.reset_highlights()

In [47]:
display(gui, debug_view)

Output()

In [144]:
gui.puzzle = puzzle
gui.targets = set()
gui.anchors = set()
gui.links = set()
gui.status = "..."

In [145]:
links = set(search_hard_31(puzzle)) | set(search_hard_2(puzzle))
searching = search_chains(puzzle, links, match_usefull, connecting=softlink_123)

In [242]:
it = next(searching)
pprint(it)
gui.links = set(it)
# gui.anchors = set(it.edges)
# gui.links = {it}
# gui.anchors = set(it)

(HLink((Node(zone=Zone(box=3, row=Ellipsis, col=7), digits=Digits({2})), Node(zone=Zone(box=6, row=4, col=7), digits=Digits({2})),)),
 SLink((Node(zone=Zone(box=6, row=4, col=7), digits=Digits({2})), Node(zone=Zone(box=4, row=4, col=Ellipsis), digits=Digits({2})),)),
 HLink((Node(zone=Zone(box=4, row=4, col=Ellipsis), digits=Digits({2})), Node(zone=Zone(box=4, row=6, col=Ellipsis), digits=Digits({2})),)))


In [58]:
task = asyncio.create_task(
    solve_ui(
        puzzle,
        naked_singles,
        onceolver(hidden_singles),
        onceolver(naked_multiples),
        onceolver(hidden_multiples),
        onceolver(locked_triplets),
        onceolver(chains_("123")),
        # randomchoice,
        filtout={"naked_singles", "hidden_singles", "naked_multiples", "hidden_multiples"},
    )
)

In [ ]:
task

In [319]:
task.cancel()  # it breaks something

False